# Análisis de datasets de incendios

Este notebook recorre los cuatro CSV de las carpetas `DL_FIRE_*` y muestra estadísticas de cada dataset. Ejecútalo desde la carpeta del proyecto.

## Preparación y búsqueda de archivos

In [ ]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DATASETS = sorted(ROOT.glob("DL_FIRE_*/fire_archive_*.csv"))

if not DATASETS:
    raise FileNotFoundError(
        f"No se encontraron archivos CSV en {ROOT}. Ejecuta el notebook desde la carpeta del proyecto."
    )

print(f"Datasets encontrados: {len(DATASETS)}")
for dataset_path in DATASETS:
    print(dataset_path.relative_to(ROOT))

## Información general

In [ ]:
def show_general_info(df):
    print("Columnas:", df.columns.tolist())
    print("Registros:", len(df))
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nPrimeras filas:")
    print(df.head())
    print("\nValores nulos:")
    print(df.isnull().sum())

## Distribuciones y niveles de confianza

In [ ]:
def show_distributions(df):
    print("\nSatélites:")
    print(df["satellite"].value_counts())
    print("\nConfianza:")
    print(df["confidence"].value_counts())
    print("\nTipos de detección:")
    print(df["type"].value_counts())
    print("\nVersiones:")
    print(df["version"].value_counts())
    print("\nDetecciones diurnas/nocturnas:")
    print(df["daynight"].value_counts())

## Periodo y potencia radiativa (FRP)

In [ ]:
def show_period_and_frp(df):
    print("\nPeriodo real:")
    print(df["acq_date"].min(), df["acq_date"].max())
    print("\nFRP:")
    print(df["frp"].describe())
    print("\nValores FRP iguales a cero:")
    print((df["frp"] == 0).sum())

## Duplicados y detecciones diarias

In [ ]:
def show_data_quality_and_daily_counts(df):
    print("\nDuplicados exactos:")
    print(df.duplicated().sum())

    print("\nDetecciones por día:")
    daily = df.groupby("acq_date").size()
    print("Días con detecciones:", len(daily))
    print("Mínimo diario:", daily.min())
    print("Máximo diario:", daily.max())

    print("\nDía con más detecciones:")
    print(daily.idxmax(), daily.max())

## Ejecutar análisis para los cuatro datasets

Cada archivo se carga y analiza por separado para evitar mantener los cuatro DataFrames en memoria al mismo tiempo.

In [ ]:
for dataset_path in DATASETS:
    df = pd.read_csv(dataset_path)

    print(f"\n{'=' * 60}")
    print(f"Dataset: {dataset_path.parent.name} / {dataset_path.name}")

    show_general_info(df)
    show_distributions(df)
    show_period_and_frp(df)
    show_data_quality_and_daily_counts(df)

## Fechas sin detecciones

In [ ]:
for dataset_path in DATASETS:
    df = pd.read_csv(dataset_path)
    all_days = pd.date_range("2024-01-01", "2024-12-31")

    observed_days = pd.DatetimeIndex(
        pd.to_datetime(df["acq_date"].unique())
    )

    missing_days = all_days.difference(observed_days)

    print(f"\nDataset: {dataset_path.parent.name} / {dataset_path.name}")
    print("\nFechas sin detecciones:")
    print(missing_days.strftime("%Y-%m-%d").tolist())